In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import itertools
from pathlib import Path
import re

import h5py
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
from src.data import add_metadata_features
from src.stimuli import POD_dict

In [ ]:
sns.set_context(font_scale=1.5)

In [ ]:
eois_path = "outputs/trf_eois/eois.csv"
all_epoch_paths = list(Path("outputs/epochs_preprocessed").glob("*_epo.fif"))
epoched_phoneme_path = "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-all.h5"
epoched_phoneme_onset_path = "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-onsets.h5"
pval_threshold = 1e-3
outdir = "."

In [ ]:
eoi_df = pd.read_csv(eois_path, index_col=["subject", "electrode", "feature_block"])
# only retain feature blocks showing positive UV
eoi_df = eoi_df[eoi_df.unique_variance > pval_threshold]
eoi_df

In [ ]:
eoi_df.groupby("feature_block").size()

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path))
    try:
        epochs[subject_name].metadata = add_metadata_features(epochs[subject_name].metadata)
    except AssertionError:
        del epochs[subject_name]
        continue

In [ ]:
all_md = pd.concat([epochs[subject_name].metadata for subject_name in epochs])
g = sns.displot(all_md.behavior_linear)

# quantile cut
all_md["behavior_quantile"], bins = pd.cut(all_md.behavior_linear, 4, retbins=True)
for qleft, qright in zip(bins[:-1], bins[1:]):
    g.ax.axvline(qleft, color="black", linestyle="--")

## Plots

In [ ]:
def make_epochs_df(feature_block, baseline=None):
    all_plot_epochs, all_plot_epochs_df = {}, {}

    epochs_ = epochs
    if baseline is not None:
        epochs_ = {subject: epochs[subject].copy().apply_baseline(baseline) for subject in epochs}
    for subject, channel in eoi_df.loc[(slice(None), slice(None), feature_block)].index:
        plot_epochs = epochs_[subject]
        plot_epochs_df = plot_epochs.metadata.loc[plot_epochs.selection]
        plot_epochs_df["mismatch"] = plot_epochs_df.mismatch.map({-1: False, 1: True})
        plot_epochs_df["mismatch_left_right"] = plot_epochs_df["mismatch_left_right"].map({-1: "left", 1: "right"})
        plot_epochs_df["lexical_evidence_cue"] = plot_epochs_df["lexical_evidence_cue"].map({-1: "left", 1: "right"})
        plot_epochs_df["categorical_acoustic_cue"] = plot_epochs_df["categorical_acoustic_cue"].map({-1: "left", 1: "right"})
        plot_epochs_df["categorical_behavior"] = plot_epochs_df["behavior_categorical"].map({-1: "left", 0: "neither", 1: "right"})

        all_plot_epochs[subject, channel] = plot_epochs.get_data()[:, channel, :]
        all_plot_epochs_df[subject, channel] = plot_epochs_df

    all_plot_epochs_df = pd.concat(all_plot_epochs_df, names=["subject", "channel", "epoch_idx"])
    all_plot_epochs_df["facet_label"] = all_plot_epochs_df.index.get_level_values("subject").str.cat(
        (all_plot_epochs_df.index.get_level_values("channel") + 1).astype(str), sep="_")

    return all_plot_epochs, all_plot_epochs_df

In [ ]:
def plot_epochs(feature_block, feature_name, hue, style=None,
                hue_bins=None, hue_order=None, style_order=None,
                save=True, close=True, suffix="",
                smoke_test=False, **kwargs):
    all_plot_epochs, all_plot_epochs_df = make_epochs_df(feature_block, **kwargs)

    # if hue is continuous, bin first
    if hue is not None and all_plot_epochs_df[hue].dtype == float:
        if hue_bins is not None:
            all_plot_epochs_df[hue] = pd.cut(all_plot_epochs_df[hue], hue_bins)
            if all_plot_epochs_df[hue].isna().any():
                raise ValueError(f"NaNs in {hue} after binning")
        else:
            all_plot_epochs_df[hue] = pd.cut(all_plot_epochs_df[hue], 4)

    if hue_order is None:
        hue_order = sorted(all_plot_epochs_df[hue].unique())
    cmap = sns.color_palette("tab10", len(hue_order))
    
    style_mapper = None
    if style is not None and style_order is None:
        style_order = sorted(all_plot_epochs_df[style].unique())
        assert len(style_order) <= 4
    if style_order is not None:
        style_mapper = dict(zip(style_order, ['-', '--', ':', '-.']))

    if smoke_test:
        all_plot_epochs_df = all_plot_epochs_df.sample(4)

    col_order = sorted(all_plot_epochs_df.phoneme_pair.unique())
    g = sns.FacetGrid(data=all_plot_epochs_df.reset_index(["subject", "channel"]), row="facet_label",
                      col="phoneme_pair", col_order=col_order, height=4, aspect=2,
                      gridspec_kws={"hspace": 0.55})

    def f(data, **f_kwargs):
        ax = plt.gca()

        subject = data.subject.iloc[0]
        channel = data.channel.iloc[0]
        phoneme_pair = data.phoneme_pair.iloc[0]
        
        ax.set_title(f"{subject}_{channel + 1} {phoneme_pair}")

        ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)

        times = next(iter(epochs.values())).times

        if hue is None:
            hue_order_ = [-1]
        else:
            hue_order_ = hue_order

        if style is None:
            style_order_ = [-1]
        else:
            style_order_ = style_order
        
        seen_hues = set()
        seen_styles = set()
        for hue_level in hue_order_:
            for style_level in style_order_:
                md_ij = data
                if hue_level != -1:
                    md_ij = md_ij[md_ij[hue] == hue_level]
                if style_level != -1:
                    md_ij = md_ij[md_ij[style] == style_level]
                eps_ij_idxs = md_ij.index

                color = cmap[hue_order.index(hue_level)] if cmap is not None else None
                linestyle = style_mapper[style_level] if style_mapper is not None else None
                label = f"{hue_level} {style_level}" if style_level != -1 else hue_level

                # eps_ij_idxs = eps_ij_idxs_[eps_ij_idxs_ < all_plot_epochs[subject, channel].shape[0]]
                if len(eps_ij_idxs) == 0:
                    print(f"yerp, no epochs for {subject} {channel} {phoneme_pair} {hue_level} {style_level}")

                    # still add a legend handle
                    ax.plot([], [], label=label, color=color, linestyle=linestyle)
                else:
                    eps_ij = all_plot_epochs[subject, channel][eps_ij_idxs]
                    ax.plot(times, eps_ij.mean(axis=0),
                            label=label,
                            color=color, linestyle=linestyle)
                    seen_hues.add(hue_level)

                    # fillbetween with sem
                    sem = eps_ij.std(axis=0) / np.sqrt(eps_ij.shape[0])
                    ax.fill_between(times, eps_ij.mean(axis=0) - sem, eps_ij.mean(axis=0) + sem, alpha=0.3,
                                    label=None, color=color)

        # annotate point of disambiguation
        pod = POD_dict[phoneme_pair]
        ax.axvline(pod, color="black", linestyle="dotted")

        # annotate on last column
        if data.phoneme_pair.iloc[0] == col_order[-1]:
            # add eoi annotation
            channel_uv = eoi_df.loc[(subject, channel, feature_block), "unique_variance"]
            ax.text(1.1, 0.5, f"UV={channel_uv:.4f}", transform=ax.transAxes, ha="left", va="center")

    g.map_dataframe(f)

    for axs in g.axes:
        # add legend on final axis outside of data, left-aligned to the axis edge
        axs[-1].legend(loc="center left", bbox_to_anchor=(1.1, 0.75), title=hue)

    if save:
        g.savefig(f"{outdir}/epochs-{feature_block}-{feature_name}{suffix}.pdf")
    if close:
        plt.close(g.figure)

    return g, all_plot_epochs, all_plot_epochs_df

In [ ]:
def add_timit_insets(g):
    for row, name in zip(g.axes, g.row_names):
        subject, channel_name = name.split("_")
        channel = int(channel_name) - 1

        epoch_sources = {
            "All": epoched_phoneme_path,
            "Word onset": epoched_phoneme_onset_path,
        }
        num_insets = len(epoch_sources)
        inset_width, inset_height = 0.25, 0.2
        inset_wspace = 0.025
        inset_anchor_x = 1 - num_insets * (inset_width + inset_wspace)
        inset_anchor_y = 1.2
        assert inset_anchor_x >= 0

        # compute ymin and ymax across sources for epoched phoneme response
        ys = []
        for epoch_source in epoch_sources.values():
            with h5py.File(epoch_source, "r") as f:
                phoneme_epochs = f[subject]["epochs"][:, channel, :]
                ys.append(phoneme_epochs.flatten())
        ys = np.concatenate(ys)
        ymin = np.percentile(ys, 10)
        ymax = np.percentile(ys, 90)
        del ys

        for i, (epoch_source_name, epoch_source) in enumerate(epoch_sources.items()):
            epoch_df = pd.read_hdf(epoch_source, f"{subject}/epoch_df")

            for ax, phoneme_pair in zip(row, g.col_names):
                plot_phonemes = list(phoneme_pair.upper())
                plot_epoch_dfs = [
                    epoch_df[epoch_df.epoch_label == phoneme]
                    for phoneme in plot_phonemes
                ]

                with h5py.File(epoch_source, "r") as f:
                    epoch_tmin, epoch_tmax = f[subject].attrs["epoch_tmin"], f[subject].attrs["epoch_tmax"]
                    epoch_sfreq = f[subject].attrs["sfreq"]
                    plot_epochs_ij = [
                        f[subject]["epochs"][plot_epoch_df.index, channel, :]
                        for phoneme, plot_epoch_df in zip(plot_phonemes, plot_epoch_dfs)
                    ]

                ax_inset = ax.inset_axes([inset_anchor_x + inset_wspace + i * (inset_width + inset_wspace),
                                          inset_anchor_y - inset_height,
                                          inset_width, inset_height])
                ax_inset.axvline(0, color="gray", linestyle="--", alpha=0.5)
                ax_inset.axhline(0, color="gray", linestyle="--", alpha=0.5)
                for phoneme, ph_df, ph_epochs in zip(plot_phonemes, plot_epoch_dfs, plot_epochs_ij):
                    times = np.arange(ph_epochs.shape[1]) / epoch_sfreq + epoch_tmin
                    ax_inset.plot(times, ph_epochs.mean(axis=0), label=phoneme)
                    sem = ph_epochs.std(axis=0) / np.sqrt(ph_epochs.shape[0])
                    ax_inset.fill_between(times, ph_epochs.mean(axis=0) - sem, ph_epochs.mean(axis=0) + sem, alpha=0.3)

                ax_inset.set_title(epoch_source_name, fontsize="small")
                ax_inset.set_xlim(epoch_tmin, epoch_tmax)
                ax_inset.set_ylim(ymin, ymax)

                # y tick labels on first axis only
                if i > 0:
                    ax_inset.tick_params(axis="y", labelleft=False)
                # legend and x tick labels on last axis only
                if i == num_insets - 1:
                    ax_inset.legend(loc="upper right", bbox_to_anchor=(1.5, 1.2))
                else:
                    ax_inset.tick_params(axis="x", labelbottom=False)

                # move title to left edge so insets have room
                ax.title.set_position((0.1, 1))

    return g

In [ ]:
feature_block = "behavior"
g, p_ep, p_df = plot_epochs(feature_block, "categorical_behavior",
                            hue="categorical_behavior")

In [ ]:
feature_block = "behavior"
g, p_ep, p_df = plot_epochs(feature_block, "categorical_behavior",
                            hue="categorical_behavior", baseline=(None, 0), suffix="-baselined")

In [ ]:
feature_block = "behavior"
g, p_ep, p_df = plot_epochs(feature_block, "behavior_linear",
                         hue="behavior_linear")

In [ ]:
feature_block = "behavior"
g, p_ep, p_df = plot_epochs(feature_block, "behavior_linear",
                         hue="behavior_linear", baseline=(None, 0), suffix="-baselined")

In [ ]:
feature_block = "lexical_evidence"
g, p_ep, p_df = plot_epochs(feature_block, "lexical_evidence_cue", hue="lexical_evidence_cue",
                            save=False, close=False)
g = add_timit_insets(g)
g.savefig(f"{outdir}/epochs-{feature_block}-lexical_evidence_cue.pdf")
plt.close(g.figure)

In [ ]:
feature_block = "lexical_evidence"
g, p_ep, p_df = plot_epochs(feature_block, "lexical_evidence_cue", hue="lexical_evidence_cue",
                            save=False, close=False, baseline=(None, 0))
g = add_timit_insets(g)
g.savefig(f"{outdir}/epochs-{feature_block}-lexical_evidence_cue-baselined.pdf")
plt.close(g.figure)

In [ ]:
feature_block = "mismatch"
g, p_ep, p_df = plot_epochs(feature_block, "mismatch",
                            hue="mismatch_left_right", hue_order=["left", "right"],
                            save=False, close=False)
g = add_timit_insets(g)
g.savefig(f"{outdir}/epochs-{feature_block}-mismatch.pdf")
plt.close(g.figure)

In [ ]:
feature_block = "mismatch"
g, p_ep, p_df = plot_epochs(feature_block, "mismatch",
                            hue="mismatch_left_right", hue_order=["left", "right"],
                            save=False, close=False, baseline=(None, 0))
g = add_timit_insets(g)
g.savefig(f"{outdir}/epochs-{feature_block}-mismatch-baselined.pdf")
plt.close(g.figure)

In [ ]:
feature_block = "acoustic"
g, p_ep, p_df = plot_epochs(feature_block, "acoustic",
                            hue="categorical_acoustic_cue", hue_order=["left", "right"],
                            save=False, close=False)
g = add_timit_insets(g)
g.savefig(f"{outdir}/epochs-{feature_block}-acoustic.pdf")
plt.close(g.figure)

In [ ]:
feature_block = "acoustic"
g, p_ep, p_df = plot_epochs(feature_block, "acoustic",
                            hue="categorical_acoustic_cue", hue_order=["left", "right"],
                            save=False, close=False, baseline=(None, 0))
g = add_timit_insets(g)
g.savefig(f"{outdir}/epochs-{feature_block}-acoustic-baselined.pdf")
plt.close(g.figure)